In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from pyprojroot import here

In [7]:
INTERIM_DATA_DIR = here() / "data" / "interim"
RESULTS_DATA_DIR = here() / "results" / "esda"
RAW_DATA_DIR = here() / "data" / "raw"

In [8]:
mun = gpd.read_file(RAW_DATA_DIR/"geographic/municipalities/00mun.shp")  # CRS: MEXICO_ITRF_2008_LCC (projected, meters)
pop = pd.read_csv(INTERIM_DATA_DIR/"population_density.csv", dtype={"CVEGEO": str})

In [9]:
pop25 = pop[["CVEGEO", "population"]].copy()
mun = mun.merge(pop25, on="CVEGEO", how="left")

In [10]:
missing = mun["population"].isna().sum()
if missing:
    print(f"Warning: {missing} municipalities have no population — filling with 1")
    mun["population"] = mun["population"].fillna(1)

In [11]:
mun = mun.sort_values("CVEGEO").reset_index(drop=True)
centroids = mun.geometry.centroid
coords = np.column_stack([centroids.x, centroids.y])   # shape (2478, 2)
mass    = mun["population"].values.astype(float)             # shape (2478,)

In [12]:
dist_m = cdist(coords, coords, metric="euclidean") 

In [13]:
dist_km = dist_m / 1000.0

In [14]:
with np.errstate(divide="ignore", invalid="ignore"):
    G = np.outer(mass, mass) / (dist_km ** 2)

np.fill_diagonal(G, 0.0)

In [15]:
cvegeo = mun["CVEGEO"].values

G_df = pd.DataFrame(G, index=cvegeo, columns=cvegeo)
G_df.index.name   = "origin_CVEGEO"
G_df.columns.name = "dest_CVEGEO"

G_df.to_csv(INTERIM_DATA_DIR/"gravity_matrix_2025.csv")
print("Saved → gravity_matrix_2025.csv")
print(f"Matrix shape : {G_df.shape}")
print(f"Non-zero cells: {(G > 0).sum():,}")
print(f"\nSample stats (off-diagonal):")
off = G[G > 0]
print(f"  min  : {off.min():.4f}")
print(f"  max  : {off.max():.4e}")
print(f"  mean : {off.mean():.4e}")
print(f"  median: {np.median(off):.4e}")

Saved → gravity_matrix_2025.csv
Matrix shape : (2478, 2478)
Non-zero cells: 6,138,006

Sample stats (off-diagonal):
  min  : 0.0000
  max  : 1.0007e+11
  mean : 1.3947e+06
  median: 2.7274e+03
